#### loss function 손실함수
: 모델의 예측값과 실제 정답이 얼마나 다른지 수치로 계산하는 함수(지도학습)
: 모델의 궁극적인 목표는 손실값을 줄이는 것

- 기본용어
    - Prediction: 모델이 예측한 값
    - Target: 정답
    - Loss: 예측값과 정답의 차이를 계산한 값

- 손실함수의 역할
    - 예측값과 정답의 차이를 수치로 표현
    - 모델이 얼마나 잘 예측했는지 평가할 수 있음
    - 자동미분을 통해weight와 bias를 업데이틀 할 때 사용할 기울기를 계산할 수 있음

#### 1. MSELoss(Mean Square Error Loss Function)
: 평균의 제곱 오차를 계산하는 손실함수
예측값과 정답의 차이를 제곱한 뒤 평균을 계산함

- 특징
    - 오차를 제곱하므로 오차가 클수록 더 큰 패널티(손실값)를 부여함
    - 손실값은 항상 0 이상임
    - 예측값이 정답과 같을 때 손실값은 0
    - 주로 회귀 문제에서 사용

- 사용방법
```python
loss_fn = nn.MSELoss()
loss_fn(prediction, target) 
# 순서가 바뀌어도 어차피 둘의 차이를 제곱하기에 상관없음
```

- 기본설정
    - reduction = 'mean'은 모든 원소의 제곱오차의 평균

In [1]:
import torch
import torch.nn as nn

In [4]:
pred = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)

print('예측값:', pred)
print('실제값:', target)
print('MSE Loss:', loss)

# 계산으로 검증
loss2 = ((target - pred) ** 2).mean()
print('직접 계산한 MSE Loss:', loss2)

예측값: tensor([2., 4., 6.])
실제값: tensor([1., 5., 4.])
MSE Loss: tensor(2.)
직접 계산한 MSE Loss: tensor(2.)


#### 2. L1Loss(Mean Absolute Error)
: 예측값과 정답의 차이를 절댓값을 적용한 뒤 평균 계산

- 특징
    - 오차에 절댓값 사용하여 손실값은 0 이상
    - 예측값과 정답이 같으면 손실값은 0
    - MSE 보다 큰 오차에 대한 패널티가 상대적으로 적음
    - Lasso Regression에 사용

- 사용법
```python
loss_fn = nn.L1loss()
loss = loss_fn(pred, target)
```

In [7]:
pred = torch.tensor([2., 4, 6])
target = torch. tensor([1., 5, 4])

loss_fn = nn.L1Loss()
loss = loss_fn(pred, target)

print('예측값:', pred)
print('실제값:', target)
print('L1Loss:', loss)  # MSE loss: tensor(2.) 보다 작음을 확인

# 계산으로 검증
loss2 = torch.abs(target - pred).mean()
print('직접 계산한 L1Loss:', loss2)

예측값: tensor([2., 4., 6.])
실제값: tensor([1., 5., 4.])
L1Loss: tensor(1.3333)
직접 계산한 L1Loss: tensor(1.3333)


- 큰 오차를 더 강하게 반영하고 싶다면 MSE
- 이상치의 영향을 상대적으로 적게 받고 싶으면 L1
- 오차가 많을 수록 패러티 값이 더 크기에 패러티 값을 줄이는 것을 목표로 하는 보통의 경우
    : MSE 더 많이 씀

In [9]:
prediction = torch.tensor([10., 10, 10])
target = torch.tensor([9., 8., 0])

mse_loss = nn.MSELoss()(prediction, target)
l1_loss = nn.L1Loss()(prediction, target)

print('MSE Loss:', mse_loss)
print('L1 Loss:', l1_loss)

# 오차가 큰 tensor[2]의 영향으로 MSE의 값이 훨씬 크다는 것을 확인

MSE Loss: tensor(35.)
L1 Loss: tensor(4.3333)


#### 3. BCEloss (Binary Cross Error)
: 이진분류에서 사용하는 손실 함수

**중요**
모델이 sigmoid를 통과한 확률값을 입력으로 받아야 함.

```ptyhon
nn.BCELoss()(probability, target)
```
probability 는 0과 1 사이의 확률값<br>
target은 정답. 일반적으로 0 또는 1이며, 반드시 실수형 텐서로 전달(정수 X)

- 실제 정답이 1인데 확률이 1에 가까우면 손실이 작아지고, 0에 가까워지면 손실이 커짐.
- 극단적인 값에서 Nan 발생 위험 있음

In [ ]:
probability = torch.tensor([0.9, 0.2, 0.7, 0.1])
target = torch.tensor([1., 0, 1, 0])

loss_fn = nn.BCELoss()
loss = loss_fn(probability, target)

print('예측값:', probability)
print('실제값:', target)
print('BCE Loss 값:', loss)

# BCELoss = y * log(p) + (1-y) * log(1-p)
# y는 target 값
# p는 probability 값

예측값: tensor([0.9000, 0.2000, 0.7000, 0.1000])
실제값: tensor([1., 0., 1., 0.])
BCE Loss 값: tensor(0.1976)


In [ ]:
import math
print(math.log(0.9))
print(torch.log(torch.tensor(0.9)))

-0.10536051565782628


#### 4. BCEWithLogitsLoss
: Sigmoid와 BCELoss 계산을 하나로 결합한 손실함수
- 극단적인 값에서 nan이 발생할 위험이 있는 BCE loss를 sigmoid로 보완

- 중요한 차이
    - BCEloss() : sigmoid를 적용한 확률값을 입력으로 받음
    - BCEWithLogitsLoss() : sigmoid를 적용하지 않은 Logits을 입력으로 받음

- 사용방식
```python
nn.BCEWithLogitsLoss()(logits, target)
```

- 사용하는 이유
: Sigmoid와 BCELoss계산을 결합하여 수치적으로 안정성이 높은 손실 계산이 가능
    - 극단적인 값에서 Nan 발생 위험이 적음

In [22]:
logits = torch.tensor([2., -1.5, 1, -2])
target = torch.tensor([1., 0, 1, 0])

prob = torch.sigmoid(logits) # all elements of input should be between 0 and 1
loss_fn1 = nn.BCELoss()
loss1 = loss_fn1(prob, target)
print('logits:', logits)
print('prob:', prob)
print('BCE loss after sigmoid activation:', loss1)
print()

loss_fn2 = nn.BCEWithLogitsLoss()
loss2 = loss_fn2(logits, target) # 바로 logit 입력
print('BCEWithLogitsloss:', loss2)

# 두 손실 값이 일치하는 지 확인 
# torch.allclose(비교값1, 비교값2)
print('Do they match?', torch.allclose(loss2, loss1))

logits: tensor([ 2.0000, -1.5000,  1.0000, -2.0000])
prob: tensor([0.8808, 0.1824, 0.7311, 0.1192])
BCE loss after sigmoid activation: tensor(0.1921)

BCEWithLogitsloss: tensor(0.1921)
Do they match? True


In [25]:
logits_a = torch.tensor([1000.], requires_grad= True)
target = torch.tensor([0.])

prob = torch.sigmoid(logits_a)
loss_a = nn.BCELoss()(prob, target)

loss_a.backward()
print(logits_a.grad)

logits_b = torch.tensor([1000.], requires_grad= True)
loss_b = nn.BCEWithLogitsLoss()(logits_b, target)

print(loss_a)
print(loss_b)

tensor([0.])
tensor(100., grad_fn=<BinaryCrossEntropyBackward0>)
tensor(1000., grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
